# C2 · Adding copies of a feature makes it look useless

**Type**: Break the model · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Add `k` almost identical copies of the most important feature and observe that the permutation importance of **each** copy drops toward zero (the model can use the others) while the sum stays and the impurity importance is split. Then fix it with the **group permutation**:
shuffling the correlated columns together.

## Dataset

`make_classification(2000, 8 features: 4 informative, class_sep=1, flip_y=0.05, shuffle=False)`, 60/40 split. The most important feature is column 0; the copies are `x0 + noise(0.05)`.

## Educational purpose

Understand a limit of the interpretation of tree models: with redundant features the individual importance is **not** the importance of the "information" they carry. An analyst who looks at the importances can conclude that a fundamental variable does not matter, just because copies of it exist.

## Task

1. Train `RandomForestClassifier(200, random_state=0, n_jobs=-1)` without copies and compute `perm0` (permutation importance on the test set, `n_repeats=10`, `random_state=0`) of column 0.
2. For `k` in `[1, 3, 5]` add `k` copies (`x0 + normal(0, 0.05)`, with `rng = np.random.default_rng(0)`, in both training and test), retrain and save in `table` (index `k`): `acc`, `orig_perm` (permutation of column 0), `mean_single_copy_perm` (mean of the permutations of the `k` copies), `orig_imp` (impurity importance of column 0) and `group_imp` (impurity sum of column 0 and the copies).
3. **Group permutation:** write `group_perm(model, X, y, columns, n_repeats, seed)`: baseline accuracy minus the mean accuracy after permuting **with the same permutation** all the given columns (the same shuffling of rows). For `k = 5` save `group_perm_k5` (column 0 and the five copies together).
4. Answer: `orig_perm_at_k5_is_zero` (True if `orig_perm` for `k = 5` is smaller than 0.02) and `group_recovers` (True if `group_perm_k5 > 0.8 * perm0`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X, y = make_classification(2000, n_features=8, n_informative=4, n_redundant=0, class_sep=1.0, flip_y=0.05, shuffle=False, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
rf = lambda: RandomForestClassifier(200, random_state=0, n_jobs=-1)


def with_copies(k, seed=0):
    rng = np.random.default_rng(seed)
    copies_tr = [X_tr[:, 0] + rng.normal(0, 0.05, len(X_tr)) for _ in range(k)]
    copies_te = [X_te[:, 0] + rng.normal(0, 0.05, len(X_te)) for _ in range(k)]
    return np.column_stack([X_tr] + copies_tr), np.column_stack([X_te] + copies_te)

## Your work

In [ ]:
perm0 = None

In [ ]:
table = pd.DataFrame(index=[1, 3, 5], columns=["acc", "orig_perm", "mean_single_copy_perm", "orig_imp", "group_imp"], dtype=float)
for k in table.index:
    A_tr, A_te = with_copies(k)
    ...
table.round(3)

In [ ]:
def group_perm(model, X, y, columns, n_repeats=10, seed=0):
    ...


group_perm_k5 = None
orig_perm_at_k5_is_zero = None
group_recovers = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Single permutation: `permutation_importance(f, A_te, y_te, n_repeats=10, random_state=0).importances_mean`. Group: `base = model.score(X, y)`; for each repetition `p = rng.permutation(len(X)); Xp = X.copy(); Xp[:, columns] = X[p][:, columns]`; `loss = base - model.score(Xp, y)`; mean.'):
    check(perm0 is not None and 0.15 < perm0 < 0.25, "without copies column 0 is worth about 0.2 of lost accuracy (0.15-0.25)", "permutation_importance(...).importances_mean[0]")
    check(table.notna().all().all() and table.loc[1, "orig_perm"] < 0.7 * perm0 and table.loc[3, "orig_perm"] < 0.1 * perm0 + 0.01, "with 1 copy the permutation of column 0 drops below 70%; with 3 below 15%", "the importance is split among the copies")
    check(table.loc[[3, 5], "orig_perm"].max() < 0.02 and table.loc[[3, 5], "mean_single_copy_perm"].max() < 0.02, "with 3 and 5 copies each column, alone, is worth less than 0.02: it looks useless", "the model uses the other copies")
    check(table["orig_imp"].is_monotonic_decreasing and table["group_imp"].is_monotonic_increasing, "the impurity of column 0 drops, that of the group rises: more copies = more total weight", "sum of the importances of the group")
    check(group_perm_k5 is not None and group_perm_k5 > 0.8 * perm0 and orig_perm_at_k5_is_zero is True and group_recovers is True, "permuting the group together recovers the importance (over 80% of the initial one)", "same row permutation for all the columns of the group")
done()

---
The commented solution is in `solutions/C_break_the_model/C2_correlated_features_and_permutation_importance.ipynb`. Open it after you pass the check, or after you have really tried.